# **Milestone 3**

## **Introduction**

This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

## **Initialization**

In [21]:
!pip install faiss-cpu #install FAISS

In [9]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating nowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Knowledge base successfully created


In [10]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

## **Question 01**

Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [11]:
res_q1 = zs(prompt_150, candidate_labels=labels_150)
idx_ans = res_q1["labels"].index(ans_150)
score_q1 = res_q1["scores"][idx_ans]
print(round(score_q1, 3))

0.384


## **Question 02**

Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [12]:
emb_150 = model.encode([prompt_150])
D, I = index.search(emb_150, 10)
rank_q2 = list(I[0]).index(150) + 1
print(rank_q2)

10


## **Question 03**

Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder.
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face) [https://huggingface.co/cross-encoder]

In [14]:
# Code to use a Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
docs_10 = [kb[i] for i in I[0]]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [15]:
sorted_indices = np.argsort(ce_scores)[::-1]
rank_q3 = list(sorted_indices).index(list(I[0]).index(150)) + 1
print(rank_q3)

1


## **Question 04**

Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [16]:
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

emb_42 = model.encode([prompt_42])
D_42, I_42 = index.search(emb_42, 5)
docs_5 = [kb[i] for i in I_42[0]]
concat_docs = " ".join(docs_5)

context_str = f"Context: {concat_docs} Question: {prompt_42}"
tokens_q4 = tokenizer_bert(context_str, truncation=False)
print(len(tokens_q4["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

216


## **Question 05**

Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [17]:
true_doc_150 = kb[150]
rag_str_q5 = f"Context: {true_doc_150} Question: {prompt_150}"

res_q5 = zs(rag_str_q5, candidate_labels=labels_150)
idx_ans_q5 = res_q5["labels"].index(ans_150)
score_q5 = res_q5["scores"][idx_ans_q5]
print(round(score_q5, 3))

0.989


## **Question 06**

Concept: The Danger of Bad Retrieval (Adversarial RAG)
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [18]:
bad_doc = kb[999]
bad_rag_str = f"Context: {bad_doc} Question: {prompt_150}"

res_q6 = zs(bad_rag_str, candidate_labels=labels_150)
idx_ans_q6 = res_q6["labels"].index(ans_150)
score_q6 = res_q6["scores"][idx_ans_q6]
print(round(score_q6, 3))

0.529


## **Question 07**

In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [19]:
hits = 0
for i in range(100):
    row = train.iloc[i]
    correct_ans_str = str(row[row["answer"]])
    prompt_str = str(row["prompt"])

    emb = model.encode([prompt_str])
    _, idxs = index.search(emb, 5)
    retrieved_docs = [kb[idx] for idx in idxs[0]]

    if any(correct_ans_str in doc for doc in retrieved_docs):
        hits += 1

print(round((hits / 100.0) * 100, 1))

73.0


## **Question 08**

Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [20]:
def map_at_3(truth, preds):
    if truth == preds[0]:
        return 1.0
    elif truth == preds[1]:
        return 1.0/2.0
    elif truth == preds[2]:
        return 1.0/3.0
    return 0.0

map_scores = []
letters = ["A", "B", "C", "D", "E"]

for i in range(20):
    row = train.iloc[i]
    prompt_str = str(row["prompt"])
    truth_label = str(row["answer"])
    options = [str(row[l]) for l in letters]

    emb = model.encode([prompt_str])
    _, idxs = index.search(emb, 5)
    retrieved_docs = [kb[idx] for idx in idxs[0]]

    pairs = [[prompt_str, doc] for doc in retrieved_docs]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = retrieved_docs[np.argmax(ce_scores)]

    rag_str = f"Context: {best_doc} Question: {prompt_str}"
    res = zs(rag_str, candidate_labels=options)

    top_3_letters = []
    for lbl in res["labels"][:3]:
        idx = options.index(lbl)
        top_3_letters.append(letters[idx])

    map_scores.append(map_at_3(truth_label, top_3_letters))

print(round(np.mean(map_scores), 3))

0.975
